## 07 — Pacote íntegro do dataset para runtimes GPU (v3)
Este notebook elimina a restauração de mais de 12 mil arquivos individuais do Google
Drive. Ele deve ser executado preferencialmente no mesmo runtime em que o Notebook 06
terminou, enquanto o RUN local ainda existe. O dataset aprovado é reunido em um TAR
imutável, acompanhado de contrato e SHA-256.
Este notebook não treina modelos e pode ser executado em CPU.

In [3]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, tarfile, time
import pandas as pd
from tqdm.auto import tqdm

def log_etapa(mensagem):
    print(f"[{time.strftime('%H:%M:%S')}] {mensagem}", flush=True)

def sha256_com_progresso(caminho, descricao, bloco=8*1024*1024):
    caminho=Path(caminho); total=caminho.stat().st_size; h=hashlib.sha256()
    with caminho.open('rb') as arq, tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:arq.read(bloco),b''):
            h.update(parte);barra.update(len(parte))
    return h.hexdigest()

def copiar_arquivo_com_progresso(origem,destino,descricao,bloco=8*1024*1024):
    origem,destino=Path(origem),Path(destino);destino.parent.mkdir(parents=True,exist_ok=True)
    total=origem.stat().st_size
    with origem.open('rb') as entrada,destino.open('wb') as saida, \
         tqdm(total=total,desc=descricao,unit='B',unit_scale=True) as barra:
        for parte in iter(lambda:entrada.read(bloco),b''):
            saida.write(parte);barra.update(len(parte))
    shutil.copystat(origem,destino)

def metadado_tar_deterministico(info):
    info.uid=0;info.gid=0;info.uname='';info.gname='';info.mtime=0
    info.mode=0o644
    return info

drive.mount('/content/drive',force_remount=True)
BACKUP_DIR=Path('/content/drive/MyDrive/tcc-satellite-streaks')
RUN_ID=(BACKUP_DIR/'data/synthetic_runs/LATEST_RUN_V3.txt').read_text(encoding='utf-8').strip()
RUN=Path('/content/data/synthetic_runs')/RUN_ID
DRIVE_RUN=BACKUP_DIR/'data/synthetic_runs'/RUN_ID
PACOTES_DRIVE=BACKUP_DIR/'data/yolo_packages'/RUN_ID
PACOTE_LOCAL=Path('/content')/f'{RUN_ID}_yolo_dataset_v3.tar'
CONTRATO_LOCAL=Path('/content')/f'{RUN_ID}_contrato_pacote_yolo_v3.json'
print('RUN esperado:',RUN_ID)
print('RUN local:',RUN)

Mounted at /content/drive
RUN esperado: synthetic_v3_c41d5091de56
RUN local: /content/data/synthetic_runs/synthetic_v3_c41d5091de56


## 1. Gate local antes do empacotamento
O caminho preferencial usa o RUN que já foi restaurado pelo Notebook 06. Se aquela sessão
ainda estiver ativa, mantenha EMPACOTAR_DIRETO_DO_DRIVE=False.
Se a sessão anterior tiver sido perdida, altere somente
EMPACOTAR_DIRETO_DO_DRIVE=True. Nesse modo o TAR é criado diretamente do RUN aprovado
no Drive: os arquivos não são restaurados individualmente para /content antes do
empacotamento. A leitura remota continuará demorada, mas ocorrerá uma única vez.

In [5]:
EMPACOTAR_DIRETO_DO_DRIVE=True

if RUN.is_dir():
    RUN_FONTE=RUN
    MODO_FONTE='run_local_do_runtime'
elif EMPACOTAR_DIRETO_DO_DRIVE:
    if not DRIVE_RUN.is_dir():
        raise RuntimeError(f'RUN aprovado também está ausente no Drive: {DRIVE_RUN}')
    RUN_FONTE=DRIVE_RUN
    MODO_FONTE='run_aprovado_direto_do_drive'
    print(
        'ATENÇÃO: o runtime local foi perdido. O TAR será criado diretamente do Drive, '
        'sem restauração intermediária. Esta etapa pode demorar bastante.'
    )
else:
    raise RuntimeError(
        'RUN local ausente. Se a sessão do Notebook 06 ainda estiver ativa, execute o '
        'Notebook 07 nela. Se aquela sessão foi perdida, altere apenas '
        'EMPACOTAR_DIRETO_DO_DRIVE=True e execute novamente esta seção.'
    )

obrigatorios=['manifest_geracao_v3.csv','config_geracao_v3.json','run_id.txt','resumo_validacao_v3.json']
for nome in obrigatorios:
    assert (RUN_FONTE/nome).is_file(),f'Ausente: {nome}'
df=pd.read_csv(RUN_FONTE/'manifest_geracao_v3.csv')
HASH_MANIFEST=hashlib.sha256((RUN_FONTE/'manifest_geracao_v3.csv').read_bytes()).hexdigest()
validacao=json.loads((RUN_FONTE/'resumo_validacao_v3.json').read_text(encoding='utf-8'))
assert len(df)==6000
assert validacao.get('aprovado_para_treino') is True
assert validacao.get('sha256_manifest')==HASH_MANIFEST

arquivos=[]
for nome in obrigatorios: arquivos.append(RUN_FONTE/nome)
for split in ['train','val','test']:
    esperado=set(df.loc[df.split==split,'id'])
    imagens={p.stem for p in (RUN_FONTE/'images'/split).glob('*.png')}
    labels={p.stem for p in (RUN_FONTE/'labels'/split).glob('*.txt')}
    assert imagens==esperado,f'Conjunto de imagens divergente em {split}'
    assert labels==esperado,f'Conjunto de labels divergente em {split}'
    arquivos.extend(sorted((RUN_FONTE/'images'/split).glob('*.png')))
    arquivos.extend(sorted((RUN_FONTE/'labels'/split).glob('*.txt')))
assert len(arquivos)==12004
print(json.dumps({
    'status':'RUN_FONTE_APROVADO_PARA_EMPACOTAMENTO','run_id':RUN_ID,
    'modo_fonte':MODO_FONTE,
    'sha256_manifest':HASH_MANIFEST,'amostras':len(df),'arquivos_no_pacote':len(arquivos)
},indent=2,ensure_ascii=False))

ATENÇÃO: o runtime local foi perdido. O TAR será criado diretamente do Drive, sem restauração intermediária. Esta etapa pode demorar bastante.
{
  "status": "RUN_FONTE_APROVADO_PARA_EMPACOTAMENTO",
  "run_id": "synthetic_v3_c41d5091de56",
  "modo_fonte": "run_aprovado_direto_do_drive",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "amostras": 6000,
  "arquivos_no_pacote": 12004
}


## 2. Construção local do TAR imutável
PNG e TAR já são formatos adequados para transferência; não usamos gzip porque ele
acrescentaria processamento com pouco ganho sobre PNGs já comprimidos.

In [6]:
RECONSTRUIR_PACOTE_LOCAL=False
if PACOTE_LOCAL.exists() and not RECONSTRUIR_PACOTE_LOCAL:
    if not CONTRATO_LOCAL.exists():
        raise RuntimeError('Pacote local existe sem contrato. Revise o alvo antes de reconstruir.')
    contrato=json.loads(CONTRATO_LOCAL.read_text(encoding='utf-8'))
    assert contrato['run_id']==RUN_ID and contrato['sha256_manifest']==HASH_MANIFEST
    hash_atual=sha256_com_progresso(PACOTE_LOCAL,'Verificando TAR local existente')
    assert hash_atual==contrato['sha256_pacote'],'TAR local diverge do contrato.'
    print('Pacote local válido reutilizado.')
else:
    temporario=PACOTE_LOCAL.with_suffix('.tar.partial')
    if temporario.exists(): temporario.unlink()
    log_etapa(f'Criando TAR local com {len(arquivos)} arquivos')
    inicio=time.perf_counter()
    with tarfile.open(temporario,'w') as tar:
        for arquivo in tqdm(arquivos,desc='Empacotando dataset',unit='arq'):
            relativo=arquivo.relative_to(RUN_FONTE)
            tar.add(
                arquivo,arcname=(Path(RUN_ID)/relativo).as_posix(),recursive=False,
                filter=metadado_tar_deterministico
            )
    temporario.replace(PACOTE_LOCAL)
    hash_pacote=sha256_com_progresso(PACOTE_LOCAL,'Calculando SHA-256 do TAR')
    base_contrato={
        'versao':'yolo_dataset_package_v3','run_id':RUN_ID,
        'sha256_manifest':HASH_MANIFEST,'arquivo':PACOTE_LOCAL.name,
        'sha256_pacote':hash_pacote,'tamanho_bytes':PACOTE_LOCAL.stat().st_size,
        'membros':len(arquivos),'formato':'tar_sem_compressao'
    }
    base_contrato['sha256_contrato']=hashlib.sha256(
        json.dumps(base_contrato,sort_keys=True,separators=(',',':')).encode()
    ).hexdigest()
    CONTRATO_LOCAL.write_text(json.dumps(base_contrato,indent=2),encoding='utf-8')
    contrato=base_contrato
    log_etapa(f'TAR concluído em {time.perf_counter()-inicio:.1f}s')
print(json.dumps(contrato,indent=2,ensure_ascii=False))
print('Publicação ainda bloqueada; execute primeiro o gate local abaixo.')

[21:48:52] Criando TAR local com 12004 arquivos


Empacotando dataset:   0%|          | 0/12004 [00:00<?, ?arq/s]

Calculando SHA-256 do TAR:   0%|          | 0.00/861M [00:00<?, ?B/s]

[22:04:36] TAR concluído em 944.1s
{
  "versao": "yolo_dataset_package_v3",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "arquivo": "synthetic_v3_c41d5091de56_yolo_dataset_v3.tar",
  "sha256_pacote": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434",
  "tamanho_bytes": 860989440,
  "membros": 12004,
  "formato": "tar_sem_compressao",
  "sha256_contrato": "110cacea3ba719b3c92f165e36d7eee07f97dc4dd5641f4bbec4ec380c761262"
}
Publicação ainda bloqueada; execute primeiro o gate local abaixo.


## 3. Gate local do pacote

In [7]:
contrato=json.loads(CONTRATO_LOCAL.read_text(encoding='utf-8'))
assert contrato['run_id']==RUN_ID
assert contrato['sha256_manifest']==HASH_MANIFEST
assert contrato['membros']==12004
assert PACOTE_LOCAL.stat().st_size==contrato['tamanho_bytes']
with tarfile.open(PACOTE_LOCAL,'r') as tar:
    membros=tar.getmembers()
assert len(membros)==contrato['membros']
assert all(not m.issym() and not m.islnk() for m in membros)
assert all(Path(m.name).parts[0]==RUN_ID for m in membros)
print(json.dumps({
    'status':'PACOTE_LOCAL_APROVADO_PARA_PUBLICACAO',
    'run_id':RUN_ID,'sha256_manifest':HASH_MANIFEST,
    'sha256_pacote':contrato['sha256_pacote'],
    'tamanho_bytes':contrato['tamanho_bytes'],'membros':len(membros)
},indent=2,ensure_ascii=False))

{
  "status": "PACOTE_LOCAL_APROVADO_PARA_PUBLICACAO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_pacote": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434",
  "tamanho_bytes": 860989440,
  "membros": 12004
}


In [8]:
import hashlib
import json
import tarfile
from tqdm.auto import tqdm

hashes_esperados = {}

for linha in df.itertuples():
    hashes_esperados[
        f'{RUN_ID}/images/{linha.split}/{linha.id}.png'
    ] = linha.sha256_imagem

    hashes_esperados[
        f'{RUN_ID}/labels/{linha.split}/{linha.id}.txt'
    ] = linha.sha256_label

assert len(hashes_esperados) == 12000

problemas_hash = []
hashes_verificados = 0

with tarfile.open(PACOTE_LOCAL, 'r') as tar:
    membros = tar.getmembers()
    nomes_tar = {m.name for m in membros}

    ausentes = set(hashes_esperados) - nomes_tar
    assert not ausentes, f'Arquivos ausentes no TAR: {list(ausentes)[:20]}'

    # Mantém a ordem física do TAR para evitar leituras aleatórias desnecessárias.
    alvos = [m for m in membros if m.name in hashes_esperados]
    assert len(alvos) == 12000

    for membro in tqdm(
        alvos,
        desc='Verificando conteúdo do TAR',
        unit='arq'
    ):
        nome = membro.name
        esperado = hashes_esperados[nome]
        extraido = tar.extractfile(membro)

        if extraido is None:
            problemas_hash.append(f'{nome}: não é arquivo regular')
            continue

        h = hashlib.sha256()

        for bloco in iter(lambda: extraido.read(8 * 1024 * 1024), b''):
            h.update(bloco)

        extraido.close()
        hashes_verificados += 1

        if h.hexdigest() != esperado:
            problemas_hash.append(
                f'{nome}: SHA-256 divergente do manifest'
            )

assert not problemas_hash, problemas_hash[:20]
assert hashes_verificados == 12000

print(json.dumps({
    'status': 'PACOTE_CONTEUDO_APROVADO',
    'run_id': RUN_ID,
    'sha256_manifest': HASH_MANIFEST,
    'sha256_pacote': contrato['sha256_pacote'],
    'hashes_de_conteudo_verificados': hashes_verificados,
    'problemas_encontrados': len(problemas_hash)
}, indent=2, ensure_ascii=False))

Verificando conteúdo do TAR:   0%|          | 0/12000 [00:00<?, ?arq/s]

{
  "status": "PACOTE_CONTEUDO_APROVADO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_manifest": "b635aff10e8b24d3729b4b399f50a8223e0d4373d155a7f16bf232c58d844b42",
  "sha256_pacote": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434",
  "hashes_de_conteudo_verificados": 12000,
  "problemas_encontrados": 0
}


## 4. Publicação e verificação no Drive
Altere a chave somente após conferir o gate local. A publicação usa um arquivo parcial
e só o renomeia depois da cópia completa.

In [9]:
PUBLICAR_PACOTE_NO_DRIVE=True
if not PUBLICAR_PACOTE_NO_DRIVE:
    print('Publicação bloqueada. Confira o gate e altere a chave para True.')
else:
    PACOTES_DRIVE.mkdir(parents=True,exist_ok=True)
    destino=PACOTES_DRIVE/PACOTE_LOCAL.name
    destino_parcial=destino.with_suffix(destino.suffix+'.partial')
    contrato_destino=PACOTES_DRIVE/'contrato_pacote_yolo_v3.json'
    if destino.exists():
        hash_remoto=sha256_com_progresso(destino,'Verificando TAR já existente no Drive')
        assert hash_remoto==contrato['sha256_pacote'],(
            'Já existe pacote remoto divergente. Não será sobrescrito automaticamente.'
        )
        print('TAR remoto idêntico reutilizado.')
    else:
        if destino_parcial.exists(): destino_parcial.unlink()
        copiar_arquivo_com_progresso(PACOTE_LOCAL,destino_parcial,'Publicando TAR no Drive')
        hash_parcial=sha256_com_progresso(destino_parcial,'Verificando TAR publicado')
        assert hash_parcial==contrato['sha256_pacote']
        destino_parcial.replace(destino)
    shutil.copy2(CONTRATO_LOCAL,contrato_destino)
    hash_final=sha256_com_progresso(destino,'Verificação final do TAR no Drive')
    assert hash_final==contrato['sha256_pacote']
    contrato_remoto=json.loads(contrato_destino.read_text(encoding='utf-8'))
    assert contrato_remoto==contrato
    print(json.dumps({
        'status':'BACKUP_PACOTE_YOLO_APROVADO','run_id':RUN_ID,
        'sha256_pacote':hash_final,'tamanho_bytes':destino.stat().st_size,
        'membros':contrato['membros']
    },indent=2,ensure_ascii=False))

Publicando TAR no Drive:   0%|          | 0.00/861M [00:00<?, ?B/s]

Verificando TAR publicado:   0%|          | 0.00/861M [00:00<?, ?B/s]

Verificação final do TAR no Drive:   0%|          | 0.00/861M [00:00<?, ?B/s]

{
  "status": "BACKUP_PACOTE_YOLO_APROVADO",
  "run_id": "synthetic_v3_c41d5091de56",
  "sha256_pacote": "163f46b536aa416c39c806490d6e6066b393bdb53a1e6de5911f24699a4b5434",
  "tamanho_bytes": 860989440,
  "membros": 12004
}
